Install/Imports

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

Upload + Read the Excel

In [ ]:
from google.colab import files
uploaded = files.upload()

FILE_NAME = "nc_raleigh_arrest_data_2015.xlsx"

df = pd.read_excel(FILE_NAME, sheet_name="Sheet1")
df.head()

In [ ]:
# Standardize
df.columns = df.columns.str.strip().str.lower()

# Ensure expected columns exist
expected = [
    "date","county_name","age","race","sex","type","reason_for_stop",
    "search_conducted","citation_issued","arrest_made","warning_issued","frisk_performed"
]
missing = [c for c in expected if c not in df.columns]
print("Missing columns:", missing)

# Clean key fields
df = df.dropna(subset=["race", "sex"])  # keep rows where group info exists
df["race"] = df["race"].str.strip().str.lower()
df["sex"]  = df["sex"].str.strip().str.lower()

# Parse date
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["month"] = df["date"].dt.month
df["dayofweek"] = df["date"].dt.dayofweek

In [ ]:
TARGET = "search_conducted"   # change to: "citation_issued" or "arrest_made"

# Make sure target is 0/1
df[TARGET] = df[TARGET].astype(int)

df["race_sex"] = df["race"] + " | " + df["sex"]

def group_rate(data, group_col, target_col):
    out = (data.groupby(group_col)[target_col]
           .agg(rate="mean", n="size")
           .sort_values(["rate","n"], ascending=[False, False]))
    out["rate"] = (out["rate"] * 100).round(2)
    return out

print("Rate by race (%):")
display(group_rate(df, "race", TARGET))

print("Rate by sex (%):")
display(group_rate(df, "sex", TARGET))

print("Rate by race × sex (%):")
display(group_rate(df, "race_sex", TARGET).head(15))

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset=["driver_race", "outcome", "stop_reason"])

In [ ]:
df["outcome"].value_counts().plot(kind="bar")
plt.title("Distribution of Traffic Stop Outcomes")
plt.xlabel("Outcome")
plt.ylabel("Count")
plt.show()

In [ ]:
df["driver_race"].value_counts().plot(kind="bar")
plt.title("Traffic Stops by Driver Race")
plt.xlabel("Driver Race")
plt.ylabel("Count")
plt.show()

In [ ]:
pd.crosstab(df["driver_race"], df["outcome"], normalize="index") * 100

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols].agg(["mean", "median", "std"])

In [ ]:
df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25)

In [ ]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df["driver_race"], df["outcome"])

chi2, p, dof, expected = chi2_contingency(contingency_table)

print("Chi-square statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)

In [ ]:
df_encoded = pd.get_dummies(df[["driver_race", "stop_reason", "search_conducted"]], drop_first=True)

corr_matrix = df_encoded.corr()

plt.figure(figsize=(12,8))
sns.heatmap(corr_matrix, cmap="coolwarm")
plt.title("Correlation Matrix for Independent Variables")
plt.show()

In [ ]:
df["outcome"].value_counts(normalize=True) * 100

In [ ]:
df["arrest_made"] = df["outcome"].apply(lambda x: 1 if x == "arrest" else 0)

df["arrest_made"].value_counts(normalize=True) * 100

In [ ]:
df[numeric_cols].boxplot(figsize=(12,6))
plt.title("Boxplots for Numeric Variables")
plt.xticks(rotation=45)
plt.show()

In [ ]:
df_model = df.dropna()

df_model.to_csv("cleaned_nc_traffic_stops.csv", index=False)